In [45]:
import os
from typing import List, Dict, Union
from datetime import datetime
from tqdm import tqdm
from collections import defaultdict
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    Docx2txtLoader,
    TextLoader,
    UnstructuredMarkdownLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

def load_and_store_documents(input_path: str, db_path: str = "./vector_db") -> Dict[str, int]:
    """
    Load documents from a file or folder, embed them, and store in a vector database.
    
    Args:
    input_path (str): Path to a file or folder containing documents.
    db_path (str): Path to store the vector database.
    
    Returns:
    Dict[str, int]: A dictionary containing statistics about processed files and chunks.
    """
    # Initialize embeddings and vector store
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)
    
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        length_function=len
    )
    
    # Determine if input_path is a file or directory
    if os.path.isfile(input_path):
        files = [input_path]
    elif os.path.isdir(input_path):
        files = [os.path.join(input_path, f) for f in os.listdir(input_path) if os.path.isfile(os.path.join(input_path, f))]
    else:
        raise ValueError("Invalid input path. Must be a file or directory.")
    
    # Initialize statistics
    stats = defaultdict(int)
    stats['total_files'] = len(files)
    stats['processed_files'] = 0
    stats['skipped_files'] = 0
    stats['total_chunks'] = 0
    chunks_per_file = {}
    
    # Process each file
    for file_path in tqdm(files, desc="Processing files"):
        # Check if file already exists in the database
        if file_exists_in_db(vectorstore, file_path):
            print(f"Skipping {file_path} as it already exists in the database.")
            stats['skipped_files'] += 1
            continue
        
        # Load document based on file extension
        _, ext = os.path.splitext(file_path)
        if ext.lower() == '.pdf':
            loader = PyMuPDFLoader(file_path)
        elif ext.lower() == '.docx':
            loader = Docx2txtLoader(file_path)
        elif ext.lower() in ['.txt', '.md']:
            loader = TextLoader(file_path) if ext.lower() == '.txt' else UnstructuredMarkdownLoader(file_path)
        else:
            print(f"Unsupported file type: {file_path}")
            stats['skipped_files'] += 1
            continue
        
        data = loader.load()
        
        # Split documents
        splits = text_splitter.split_documents(data)
        
        # Add metadata
        for split in splits:
            split.metadata = {"source": os.path.basename(file_path), "upload_date": datetime.now().isoformat()}
        
        # Add to vector store
        vectorstore.add_documents(splits)
        
        # Update statistics
        stats['processed_files'] += 1
        stats['total_chunks'] += len(splits)
        chunks_per_file[os.path.basename(file_path)] = len(splits)
    
    # Persist the database
    #vectorstore.persist()
    print(f"Vector database saved to {db_path}")
    
    # Add chunks per file to stats
    stats['chunks_per_file'] = chunks_per_file
    
    return dict(stats)

def file_exists_in_db(vectorstore: Chroma, file_path: str) -> bool:
    """Check if a file already exists in the database."""
    results = vectorstore.get(where={"source": os.path.basename(file_path)})
    return len(results['ids']) > 0

def list_files_in_db(db_path: str) -> List[Dict[str, Union[str, datetime, int]]]:
    """
    List all files in the database along with their upload dates and chunk counts.
    
    Args:
    db_path (str): Path to the vector database.
    
    Returns:
    List[Dict[str, Union[str, datetime, int]]]: List of dictionaries containing file names, upload dates, and chunk counts.
    """
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)
    
    all_metadata = vectorstore.get()['metadatas']
    unique_files = {}
    
    for metadata in all_metadata:
        file_name = metadata['source']
        upload_date = datetime.fromisoformat(metadata['upload_date'])
        
        if file_name not in unique_files:
            unique_files[file_name] = {'file_name': file_name, 'upload_date': upload_date, 'chunk_count': 1}
        else:
            unique_files[file_name]['chunk_count'] += 1
            if upload_date > unique_files[file_name]['upload_date']:
                unique_files[file_name]['upload_date'] = upload_date
    
    return list(unique_files.values())

In [48]:
load_and_store_documents("/home/ubuntu/PsycoGPT/data/kb", "mydb")

Processing files:  75%|███████▌  | 15/20 [04:21<00:32,  6.60s/it]

Skipping /home/ubuntu/PsycoGPT/data/kb/Tara Brach - Desire and Addiction Part 2.pdf as it already exists in the database.


Processing files:  85%|████████▌ | 17/20 [06:21<01:33, 31.24s/it]

Skipping /home/ubuntu/PsycoGPT/data/kb/IFS_ Demonstration_.docx as it already exists in the database.


Processing files: 100%|██████████| 20/20 [06:28<00:00, 19.44s/it]

Vector database saved to mydb


{'total_files': 20,
 'processed_files': 18,
 'skipped_files': 2,
 'total_chunks': 5507,
 'chunks_per_file': {'IFS_ No Bad Parts.docx': 536,
  'Tara Brach - Radical Acceptance_ Embracing Your Life With the Heart of a Buddha-Bantam (2004).pdf': 968,
  'IFS_ Emotional Libration and Wounded Parts.docx': 28,
  'IFS_ Intro on Model.docx': 9,
  'Tara Brach - Fear of Aging_.docx': 56,
  '_We Learn It Too Late_ - 5 Regrets Trapping People From A....pdf': 94,
  'Bessel Van Der Kolk_ Healing Trauma without Medication_.docx': 10,
  'Copy of The-Body-Keeps-the-Score-PDF.pdf': 1689,
  'How Childhood Trauma Leads to Addiction - Gabor Maté.pdf': 11,
  'IFS_ Therapy Demonstration.docx': 26,
  'IFS_ Finding True Self.docx': 113,
  'IFS - Basic Overview.docx': 28,
  'Tara Brach - Happiness and Freedom.docx': 48,
  'Tara Brach - Fear_.docx': 51,
  'Tara Brach Desire and Addiction.pdf': 49,
  'The-Body-Keeps-the-Score-PDF.pdf': 1689,
  'Tara Brach - Limiting Beliefs.docx': 47,
  'Tara Brach - Facing Fear.